In [8]:
import pandas as pd
import numpy as np

# --- File paths ---
file_path = r"D:\ChipVista\Projects\decaying fruits\old file\DATALOG-1_.csv"
save_path = r"D:\ChipVista\Projects\decaying fruits\old file\filtered_15min.csv"

# --- Load dataset ---
df = pd.read_csv(file_path)
df = df[['mq2', 'mq4', 'Time']]

# --- Step 1: Detect hours like in your plotting code ---
hours = []
current_hour = 0
prev_minute = None

for t in df['Time']:
    try:
        mins, sec_milli = t.split(":")
        mins = int(mins)
    except:
        mins = None
    
    # Detect new hour when minutes reset (59 → 0)
    if prev_minute is not None and mins is not None and mins < prev_minute:
        current_hour += 1
    
    hours.append(current_hour)
    prev_minute = mins

df['HourIndex'] = hours

# --- Step 2: Convert Time to total minutes (including hour index) ---
def time_to_minutes(t, hour_index):
    try:
        mins, sec_milli = t.split(":")
        total = int(mins) + hour_index * 60
        return total
    except:
        return np.nan

df['TotalMinutes'] = [time_to_minutes(t, h) for t, h in zip(df['Time'], df['HourIndex'])]

# --- Step 3: Keep one reading every 15 minutes ---
fifteen_min_interval = 15
df['MinuteGroup'] = (df['TotalMinutes'] // fifteen_min_interval)

# Keep the first row of each 15-minute group
filtered_df = df.groupby('MinuteGroup').first().reset_index(drop=True)

# --- Step 4: Save final filtered file with original 3 columns only ---
filtered_df = filtered_df[['mq2', 'mq4', 'Time']]
filtered_df.to_csv(save_path, index=False)

print("✅ Filtered file saved successfully!")
print(f"📁 Saved to: {save_path}")
print(f"📊 Original readings: {len(df)} → Filtered readings: {len(filtered_df)} (1 per 15 min)")


✅ Filtered file saved successfully!
📁 Saved to: D:\ChipVista\Projects\decaying fruits\old file\filtered_15min.csv
📊 Original readings: 20156 → Filtered readings: 337 (1 per 15 min)


In [11]:
import pandas as pd

file_path = r"D:\ChipVista\Projects\decaying fruits\old file\old data.csv"
df = pd.read_csv(file_path)

print("Columns:", df.columns.tolist())
print("\nFirst 10 rows of Time column:")
print(df['Time'].head(10))


Columns: ['mq2', 'mq4', 'Time']

First 10 rows of Time column:
0    36:13.0
1    45:12.6
2    00:12.6
3    15:12.6
4    30:12.6
5    45:12.6
6    00:12.6
7    15:12.6
8    30:12.6
9    45:12.6
Name: Time, dtype: object


In [13]:
import pandas as pd

# --- File paths ---
file_path = r"D:\ChipVista\Projects\decaying fruits\old file\old data.csv"
save_path = r"D:\ChipVista\Projects\decaying fruits\old file\old_data_with_voltage.csv"

# --- Load the CSV ---
df = pd.read_csv(file_path)

# --- Calculate voltage for each sensor ---
df['V_mq2'] = (df['mq2'] / 1023.0) * 5.0
df['V_mq4'] = (df['mq4'] / 1023.0) * 5.0

# Optional: round voltages for clean Excel view
df['V_mq2'] = df['V_mq2'].round(3)
df['V_mq4'] = df['V_mq4'].round(3)

# --- Save to new file ---
df.to_csv(save_path, index=False)

print(f"✅ New file saved successfully at:\n{save_path}")
print("✅ Columns added: V_mq2 and V_mq4 (voltages)")


✅ New file saved successfully at:
D:\ChipVista\Projects\decaying fruits\old file\old_data_with_voltage.csv
✅ Columns added: V_mq2 and V_mq4 (voltages)


In [14]:
import pandas as pd

# --- File path ---
file_path = r"D:\ChipVista\Projects\decaying fruits\old file\old_data_with_voltage.csv"
save_path = r"D:\ChipVista\Projects\decaying fruits\old file\with_RS_R0.csv"

# --- Load dataset ---
df = pd.read_csv(file_path)

# --- Constants ---
RL = 5000.0  # 5k ohm load resistor
V_SUPPLY = 5.0

# --- Calculate RS for each MQ sensor ---
df['RS_mq2'] = RL * ((V_SUPPLY / df['V_mq2']) - 1)
df['RS_mq4'] = RL * ((V_SUPPLY / df['V_mq4']) - 1)

# --- Determine RO from 2nd entry (after warm-up) ---
RO_mq2 = df.loc[1, 'RS_mq2']   # 2nd row → index 1
RO_mq4 = df.loc[1, 'RS_mq4']

print(f"RO_mq2 = {RO_mq2:.2f} Ω")
print(f"RO_mq4 = {RO_mq4:.2f} Ω")

# --- Calculate RS/RO ratio ---
df['RSbyRO_mq2'] = df['RS_mq2'] / RO_mq2
df['RSbyRO_mq4'] = df['RS_mq4'] / RO_mq4

# Optional: round for readability
df = df.round({'RS_mq2': 2, 'RS_mq4': 2, 'RSbyRO_mq2': 3, 'RSbyRO_mq4': 3})

# --- Save updated file ---
df.to_csv(save_path, index=False)

print(f"✅ File saved successfully at:\n{save_path}")
print("✅ Added columns: RS_mq2, RS_mq4, RSbyRO_mq2, RSbyRO_mq4")


RO_mq2 = 1036666.67 Ω
RO_mq4 = 857068.97 Ω
✅ File saved successfully at:
D:\ChipVista\Projects\decaying fruits\old file\with_RS_R0.csv
✅ Added columns: RS_mq2, RS_mq4, RSbyRO_mq2, RSbyRO_mq4
